# ✅ Calc Groups Cathedral — Check & Grade

> 🏗️ Build: **2026-05-29 12:41:15** &nbsp;·&nbsp; if you don't see this stamp after re-upload, close the notebook tab and reopen it.

## How to use this notebook

1. Click **▶️ Run all** in the toolbar.
2. Wait ~30 seconds — the model is refreshed and your 12 measures are graded.
3. Read the **grading panel** that appears below each step.

> 🕵️ The code is hidden on purpose: you don't need to read it, just run it.
> (If you're curious, click the `…` next to any cell → **Show input**.)

The checker will:
- Verify each of the 12 measures (`M_01_Current` … `M_12_DistinctCustomers`) exists on the `Sales` table.
- Evaluate each one in 3 filter contexts and compare to the canonical answer.
- Score **correctness** + **elegance** (shorter DAX, less nesting → more points).
- Log every check to `Cathedral_EH.CathedralEvents`.
- If all 12 pass → unlock the **final challenge**.


## Step 1 — Setup


In [ ]:
import subprocess, sys, importlib, json, uuid, math, re, time, getpass
import datetime as dt
import pandas as pd

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "--disable-pip-version-check", "PyJWT>=2.6.0"],
               check=False, capture_output=True)
try:
    import sempy_labs as labs
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--disable-pip-version-check", "semantic-link-labs"],
                   check=True, capture_output=True)
    importlib.invalidate_caches()
    import sempy_labs as labs
import sempy.fabric as fabric

# --- Silence sempy / .NET chatter so the grading panel stays readable ---
import logging, warnings
warnings.filterwarnings("ignore")
for name in ("sempy", "sempy.fabric", "sempy_labs", "Microsoft", "py4j"):
    logging.getLogger(name).setLevel(logging.ERROR)
# sempy prints the "'[V]'... serialized as string" line via the root logger.
logging.getLogger().setLevel(logging.ERROR)

MODEL_NAME  = "Cathedral_Model"
LAKEHOUSE   = "Cathedral_LH"
EH_CLUSTER  = "https://trd-z9b3f5xvzm87f8c2kd.z6.kusto.fabric.microsoft.com"
EH_DATABASE = "Cathedral_EH"
EH_TABLE    = "CathedralEvents"
HOST_TABLE  = "Sales"   # measures expected on this table

try:
    WORKSPACE_ID = fabric.get_notebook_workspace_id()
except Exception:
    WORKSPACE_ID = None

SESSION_ID = str(uuid.uuid4())
try:    PLAYER_ID = getpass.getuser()
except: PLAYER_ID = "anonymous"

print(f"🏢 Workspace : {WORKSPACE_ID}")
print(f"🎮 Session   : {SESSION_ID}")
print(f"🧑 Player    : {PLAYER_ID}")

# Refresh Direct Lake before grading (idempotent).
try:
    fabric.refresh_dataset(dataset=MODEL_NAME, workspace=WORKSPACE_ID, refresh_type="full")
    print("✅ Model refreshed")
except Exception as e:
    print(f"⚠️ refresh: {type(e).__name__}: {str(e)[:200]}")


## Step 2 — Helpers (DAX runner, elegance scorer, KQL logger)


In [ ]:
def _norm_dax(expr: str) -> str:
    return re.sub(r"\s+", " ", expr.strip())

import os, contextlib, io
@contextlib.contextmanager
def _muted():
    '''Silence stdout+stderr at the OS level — needed to swallow the
    .NET '[V]' serialization warnings that bypass Python logging.'''
    try:
        so_fd = os.dup(1); se_fd = os.dup(2)
        devnull = os.open(os.devnull, os.O_WRONLY)
        os.dup2(devnull, 1); os.dup2(devnull, 2)
        os.close(devnull)
        yield
    finally:
        try: os.dup2(so_fd, 1); os.close(so_fd)
        except Exception: pass
        try: os.dup2(se_fd, 2); os.close(se_fd)
        except Exception: pass

def run_dax_scalar(expr: str, ctx_filters):
    '''Wrap a scalar expression in EVALUATE ROW(CALCULATE(...)). Returns float or None.'''
    flt = ", " + ", ".join(ctx_filters) if ctx_filters else ""
    q   = f"EVALUATE ROW(\"V\", CALCULATE({expr}{flt}))"
    try:
        with _muted():
            df = fabric.evaluate_dax(dataset=MODEL_NAME, workspace=WORKSPACE_ID, dax_string=q)
        if df.empty: return None
        v = df.iloc[0, 0]
        return None if v is None else float(v)
    except Exception as e:
        return None

def elegance_score(dax: str) -> float:
    s = _norm_dax(dax)
    char_pen = max(0, len(s) - 60) * 0.4
    calc = len(re.findall(r"\bCALCULATE\b", s, re.IGNORECASE))
    filt = len(re.findall(r"\bFILTER\b",    s, re.IGNORECASE))
    sumx = len(re.findall(r"\bSUMX\b",      s, re.IGNORECASE))
    nest_pen = max(0, calc - 1) * 8 + filt * 6 + sumx * 4
    return max(0.0, round(100 - char_pen - nest_pen, 1))

def values_close(a, b, rel=1e-4, abs_tol=1e-2) -> bool:
    if a is None or b is None: return False
    return math.isclose(a, b, rel_tol=rel, abs_tol=abs_tol)

def get_measure_expression(measure_name: str) -> str | None:
    '''Read the user's DAX expression for a measure (from INFO.MEASURES()).'''
    q = (
        "EVALUATE SELECTCOLUMNS(INFO.MEASURES(), \"Name\", [Name], \"Expr\", [Expression])"
    )
    try:
        with _muted():
            df = fabric.evaluate_dax(dataset=MODEL_NAME, workspace=WORKSPACE_ID, dax_string=q)
        for _, r in df.iterrows():
            if str(r.iloc[0]) == measure_name:
                return str(r.iloc[1])
    except Exception:
        pass
    return None

def _kql_token():
    try:
        import notebookutils
        return notebookutils.credentials.getToken("kusto")
    except Exception:
        pass
    try:
        import mssparkutils
        return mssparkutils.credentials.getToken("kusto")
    except Exception:
        pass
    return None

def log_event(event_type, pillar_id, pillar_key, pass_fail, elegance, rank, duration_s, dax_len):
    import requests
    ts  = dt.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%fZ")
    eid = str(uuid.uuid4())
    row = (f'"{eid}","{ts}","{SESSION_ID}","{PLAYER_ID}","{event_type}",'
           f'{pillar_id},"{pillar_key}",0,0,"{pass_fail}",1,0,{elegance},"{rank}",{duration_s:.3f},{dax_len}')
    csl = f".ingest inline into table {EH_TABLE} <|\n{row}"
    try:
        tok = _kql_token()
        if not tok: return
        requests.post(f"{EH_CLUSTER}/v1/rest/mgmt",
                      headers={"Authorization": f"Bearer {tok}",
                               "Content-Type": "application/json"},
                      json={"db": EH_DATABASE, "csl": csl}, timeout=10)
    except Exception:
        pass

print("✅ Helpers loaded.")


## Step 3 — Pillar definitions (canonical answers)


In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple

BASE = "[Sales Amount Seed]"

CTX_2024_FULL = ("'Date'[Year]=2024",)
CTX_2025_JUN  = ("'Date'[Year]=2025", "'Date'[MonthNum]=6")
CTX_2024_EUN  = ("'Date'[Year]=2024", "Customer[Region]=\"EU-North\"")
CTX_2025_FULL = ("'Date'[Year]=2025",)
CTX_2024_DEC  = ("'Date'[Year]=2024", "'Date'[MonthNum]=12")
CTX_2024_JAN  = ("'Date'[Year]=2024", "'Date'[MonthNum]=1")

@dataclass
class Pillar:
    id: int
    key: str
    measure_name: str
    title: str
    canonical_dax: str
    contexts: List[Tuple[str, ...]]
    unit: str = "number"

PILLARS = [
    Pillar(1,  "Current",          "M_01_Current",          "Current Sales",
           BASE, [CTX_2024_FULL, CTX_2025_JUN, CTX_2024_EUN]),
    Pillar(2,  "LastYear",         "M_02_LastYear",         "Sales LY",
           f"CALCULATE({BASE}, SAMEPERIODLASTYEAR('Date'[Date]))",
           [CTX_2024_FULL, CTX_2025_JUN, CTX_2024_EUN]),
    Pillar(3,  "YoY",              "M_03_YoY",              "Sales YoY (abs)",
           f"{BASE} - CALCULATE({BASE}, SAMEPERIODLASTYEAR('Date'[Date]))",
           [CTX_2024_FULL, CTX_2025_JUN, CTX_2024_EUN]),
    Pillar(4,  "YoYPct",           "M_04_YoYPct",           "Sales YoY %",
           f"DIVIDE({BASE} - CALCULATE({BASE}, SAMEPERIODLASTYEAR('Date'[Date])), "
           f"CALCULATE({BASE}, SAMEPERIODLASTYEAR('Date'[Date])))",
           [CTX_2024_FULL, CTX_2025_JUN, CTX_2024_EUN], unit="percent"),
    Pillar(5,  "YTD",              "M_05_YTD",              "Sales YTD",
           f"CALCULATE({BASE}, DATESYTD('Date'[Date]))",
           [CTX_2024_FULL, CTX_2025_JUN, CTX_2024_EUN]),
    Pillar(6,  "MTD",              "M_06_MTD",              "Sales MTD",
           f"CALCULATE({BASE}, DATESMTD('Date'[Date]))",
           [CTX_2025_JUN, CTX_2024_FULL, CTX_2024_EUN]),
    Pillar(7,  "QTD",              "M_07_QTD",              "Sales QTD",
           f"CALCULATE({BASE}, DATESQTD('Date'[Date]))",
           [CTX_2025_JUN, CTX_2024_FULL, CTX_2024_EUN]),
    Pillar(8,  "Rolling12",        "M_08_Rolling12",        "Rolling 12-month",
           f"CALCULATE({BASE}, DATESINPERIOD('Date'[Date], MAX('Date'[Date]), -12, MONTH))",
           [CTX_2024_FULL, CTX_2025_JUN, CTX_2025_FULL]),
    Pillar(9,  "BestMonth",        "M_09_BestMonth",        "Best Month Value",
           f"MAXX(VALUES('Date'[MonthNum]), {BASE})",
           [CTX_2024_FULL, CTX_2025_FULL, CTX_2024_EUN]),
    Pillar(10, "PctOfYear",        "M_10_PctOfYear",        "% of Year",
           f"DIVIDE({BASE}, CALCULATE({BASE}, ALL('Date'[MonthNum], 'Date'[MonthName], "
           f"'Date'[DayOfMonth], 'Date'[DayName], 'Date'[Date], 'Date'[DateKey], "
           f"'Date'[IsWeekend], 'Date'[Quarter])))",
           [CTX_2025_JUN, CTX_2024_DEC, CTX_2024_JAN], unit="percent"),
    Pillar(11, "AvgDailySales",    "M_11_AvgDailySales",    "Average Daily Sales",
           f"AVERAGEX(VALUES('Date'[Date]), {BASE})",
           [CTX_2024_FULL, CTX_2025_JUN, CTX_2024_EUN]),
    Pillar(12, "DistinctCustomers", "M_12_DistinctCustomers", "Distinct Customers",
           "DISTINCTCOUNT(Sales[CustomerKey])",
           [CTX_2024_FULL, CTX_2025_JUN, CTX_2024_EUN], unit="integer"),
]
PILLAR_BY_ID = {p.id: p for p in PILLARS}

RANKS = [(0, "Stonemason"), (300, "Apprentice"), (700, "Journeyman"),
         (1100, "Architect"), (1500, "Master Architect"), (1800, "Cathedral Builder")]

def rank_for(total: float) -> str:
    r = "Stonemason"
    for thr, name in RANKS:
        if total >= thr: r = name
    return r

print(f"📜 {len(PILLARS)} pillars loaded.")


## Step 4 — Grade all 12 measures


In [ ]:
def check_all():
    print("┌─ 🏛️  Cathedral Check ─────────────────────────────────────────────")
    results = []
    total_score = 0.0
    for p in PILLARS:
        t0 = time.time()
        user_expr = get_measure_expression(p.measure_name)
        if not user_expr:
            print(f"│ ⬜ #{p.id:2d} {p.measure_name:<26s}  MISSING")
            results.append({"pillar": p.id, "status": "MISSING", "score": 0})
            log_event("check", p.id, p.key, "MISSING", 0, rank_for(total_score), time.time()-t0, 0)
            continue

        # Evaluate canonical vs user measure in each context.
        all_pass = True
        for ctx in p.contexts:
            exp = run_dax_scalar(p.canonical_dax, list(ctx))
            got = run_dax_scalar(f"[{p.measure_name}]", list(ctx))
            if not values_close(got, exp):
                all_pass = False
                break

        if all_pass:
            eleg  = elegance_score(user_expr)
            score = 50 + eleg * 0.5
            total_score += score
            print(f"│ 🟢 #{p.id:2d} {p.measure_name:<26s}  score={score:5.1f}  elegance={eleg:5.1f}")
            results.append({"pillar": p.id, "status": "PASS", "score": score, "elegance": eleg})
            log_event("check", p.id, p.key, "PASS", eleg, rank_for(total_score), time.time()-t0, len(user_expr))
        else:
            print(f"│ 🔴 #{p.id:2d} {p.measure_name:<26s}  WRONG result")
            results.append({"pillar": p.id, "status": "FAIL", "score": 0})
            log_event("check", p.id, p.key, "FAIL", 0, rank_for(total_score), time.time()-t0, len(user_expr))

    rank = rank_for(total_score)
    passed = sum(1 for r in results if r["status"] == "PASS")
    print("├──────────────────────────────────────────────────────────────────")
    print(f"│  Passed       : {passed} / {len(PILLARS)}")
    print(f"│  Total score  : {total_score:.1f}")
    print(f"│  Architect    : {rank}")
    print("└──────────────────────────────────────────────────────────────────")

    if passed == len(PILLARS):
        print()
        print("🎉🎉🎉  ALL 12 PILLARS PASSED  🎉🎉🎉")
        print("       Scroll down to unlock the FINAL CHALLENGE 👇")
    else:
        missing = [r["pillar"] for r in results if r["status"] != "PASS"]
        print()
        print(f"📌 Pillars still to complete: {missing}")
        print("   Go back to Cathedral_Model and fix/add the measures, then re-run check_all().")

    return results

results = check_all()


---
## 🏆 Final Challenge — The Calculation Group

> _Read this section **only** after all 12 pillars are 🟢._

You just wrote **12 measures**. Look at them: most of them (11 out of 12) are just
`CALCULATE([Sales Amount Seed], <time-intel function>)`. Different wrappers — **same
base measure**. That repetition is a smell, and the cure has a name: **Calculation Groups**.

A calculation group is a single table that **applies a transformation to any measure
you reference inside it**. Instead of 11 time-intel measures, you write **one base
measure + 11 calculation items**. (`M_12_DistinctCustomers` stays standalone —
it's a different aggregation, not a time-intel transformation.) The dashboard then uses something like:

```dax
CALCULATE([Sales Amount Seed], 'Time Intelligence'[Calc] = "YTD")
```

…and the calc item rewrites the measure on the fly.

### 📚 Reference
- **DAX Patterns – Calculation groups** (Marco Russo / Alberto Ferrari):
  <https://www.daxpatterns.com/calculation-groups/>
- **Microsoft Learn – Calculation groups**:
  <https://learn.microsoft.com/power-bi/transform-model/calculation-groups>

### 🛠️ Build it in the web modeler

1. Open **`Cathedral_Model`** → **Open data model**.
2. In the ribbon: **Calculation group** (📐 icon in the Calculations group).
3. Name the calc group **`Time Intelligence`** and the column **`Calc`**.
4. Add **11 calculation items** with these **exact** names and expressions:

| Item name           | Expression                                                                  |
|---------------------|-----------------------------------------------------------------------------|
| `Current`           | `SELECTEDMEASURE()`                                                         |
| `LastYear`          | `CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Date'[Date]))`            |
| `YoY`               | `SELECTEDMEASURE() - CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Date'[Date]))` |
| `YoYPct`            | `DIVIDE(SELECTEDMEASURE() - CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Date'[Date])), CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Date'[Date])))` |
| `YTD`               | `CALCULATE(SELECTEDMEASURE(), DATESYTD('Date'[Date]))`                      |
| `MTD`               | `CALCULATE(SELECTEDMEASURE(), DATESMTD('Date'[Date]))`                      |
| `QTD`               | `CALCULATE(SELECTEDMEASURE(), DATESQTD('Date'[Date]))`                      |
| `Rolling12`         | `CALCULATE(SELECTEDMEASURE(), DATESINPERIOD('Date'[Date], MAX('Date'[Date]), -12, MONTH))` |
| `BestMonth`         | `MAXX(VALUES('Date'[MonthNum]), SELECTEDMEASURE())`                         |
| `PctOfYear`         | `DIVIDE(SELECTEDMEASURE(), CALCULATE(SELECTEDMEASURE(), ALL('Date')))`      |
| `AvgDailySales`     | `AVERAGEX(VALUES('Date'[Date]), SELECTEDMEASURE())`                         |

> ⚠️ **Note**: `M_12_DistinctCustomers` is **not** a time-intelligence transformation —
> it's a different aggregation on a different column. It stays as a standalone
> measure on `Sales`. The calc group has **11 items**.

> 💡 Notice `SELECTEDMEASURE()` — this is the magic. The calc item operates on
> whatever measure you wrap with `CALCULATE([...], 'Time Intelligence'[Calc] = "...")`.
> Look at `PctOfYear` — what was 200+ characters with explicit `ALL(...)` columns
> becomes one tidy `ALL('Date')`. That's the point.

When you're done, run the cell below.


## Step 5 — Verify the Calculation Group


In [ ]:
# M_12_DistinctCustomers is NOT a time-intelligence transformation,
# it stays as a standalone measure. The calc group has 11 items.
CG_PILLARS = [p for p in PILLARS if p.key != "DistinctCustomers"]
EXPECTED_ITEMS = [p.key for p in CG_PILLARS]
CG_TABLE = "Time Intelligence"
CG_COL   = "Calc"

def check_calc_group():
    print(f"┌─ 🏆 Calculation Group Check — '{CG_TABLE}'[{CG_COL}] ──────────────")
    # 1) Discover items present
    q = f"EVALUATE VALUES('{CG_TABLE}'[{CG_COL}])"
    try:
        df = fabric.evaluate_dax(dataset=MODEL_NAME, workspace=WORKSPACE_ID, dax_string=q)
    except Exception as e:
        print(f"│ ❌ Calc group not found: {type(e).__name__}: {str(e)[:200]}")
        print(f"│    Make sure the table is named exactly '{CG_TABLE}' with column '{CG_COL}'.")
        print("└─────────────────────────────────────────────────────────────────")
        return

    present = set(df.iloc[:, 0].astype(str).tolist())
    missing = [n for n in EXPECTED_ITEMS if n not in present]
    if missing:
        print(f"│ ⬜ Missing calc items: {missing}")

    # 2) Evaluate each calc item via SELECTEDMEASURE and compare to canonical.
    total_score = 0.0
    passed = 0
    for p in CG_PILLARS:
        if p.key not in present:
            print(f"│ ⬜ #{p.id:2d} {p.key:<22s}  ITEM MISSING")
            continue
        expr = f"CALCULATE({BASE}, '{CG_TABLE}'[{CG_COL}] = \"{p.key}\")"
        all_pass = True
        for ctx in p.contexts:
            exp = run_dax_scalar(p.canonical_dax, list(ctx))
            got = run_dax_scalar(expr, list(ctx))
            if not values_close(got, exp):
                all_pass = False
                break
        if all_pass:
            passed += 1
            eleg  = elegance_score(expr)
            score = 50 + eleg * 0.5
            total_score += score
            print(f"│ 🟢 #{p.id:2d} {p.key:<22s}  score={score:5.1f}  elegance={eleg:5.1f}")
            log_event("calcgroup", p.id, p.key, "PASS", eleg, rank_for(total_score), 0.0, len(expr))
        else:
            print(f"│ 🔴 #{p.id:2d} {p.key:<22s}  WRONG result")
            log_event("calcgroup", p.id, p.key, "FAIL", 0, rank_for(total_score), 0.0, len(expr))

    rank = rank_for(total_score)
    print("├──────────────────────────────────────────────────────────────────")
    print(f"│  Passed       : {passed} / {len(CG_PILLARS)}")
    print(f"│  Total score  : {total_score:.1f}  (the elegance kicker!)")
    print(f"│  Architect    : {rank}")
    print("└──────────────────────────────────────────────────────────────────")
    if passed == len(CG_PILLARS):
        print()
        print("🏛️🏛️🏛️  CATHEDRAL BUILT  🏛️🏛️🏛️")
        print(f"       You are now a {rank}.")
        print("       One base measure × one calculation group = 12 KPIs.")
        print("       Welcome to the master path.")

check_calc_group()
